# 基盤モデルのチューニングとデプロイ


**学習目標**

1. Gemini チューニング用の JSONL ファイルを生成する方法を学ぶ
1. チューニング ジョブを開始する方法を学ぶ
1. チューニング済み LLM をデプロイしてクエリを実行する方法を学ぶ
1. チューニング済み LLM を評価する方法を学ぶ


LLM をゼロから作成するには、膨大な量のデータ、多大なコンピューティング リソース、そして専門的なスキルが必要です。このノートブックでは、チューニングによって Agent Platform Studio 上で特定のタスクや知識ドメインに合わせて Gemini 基盤モデルをカスタマイズする方法を学びます。
プロンプト設計は迅速な実験には優れていますが、トレーニング データが利用可能な場合は、モデルをチューニングすることでより高い品質を達成できます。モデルをチューニングすることで、モデルに実行させたいタスクの例に基づいてモデルの応答をカスタマイズできます。

チューニングの詳細については、[公式ドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/tuning) をご覧ください。


## セットアップ


In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
import json
import time

import evaluate
import pandas as pd
from google import genai
from google.cloud import bigquery
from google.genai import types
from IPython.display import Markdown

In [ ]:
%load_ext google.cloud.bigquery

REGION = "global"
PROJECT_ID = !(gcloud config get-value project)
PROJECT_ID = PROJECT_ID[0]
BUCKET_NAME = PROJECT_ID
BUCKET_URI = f"gs://{BUCKET_NAME}"

In [ ]:
!gcloud storage ls --buckets | grep -w $BUCKET_URI || gcloud storage buckets create --project=$PROJECT_ID --location=$REGION $BUCKET_URI

## トレーニング データ


このノートブックでは、Gen AI SDK を使用して、StackOverflow の Q&A データセットに基づいて Gemini をチューニングします。
最初のステップは、BigQuery 一般公開データセットの StackOverflow データをクエリし、`python` タグが付いた質問と、2020-01-01 以降の `accepted`（承認済み）回答のみに絞り込むことです。

データセットは 1,000 サンプルに制限し、そのうち 800 サンプルを LLM のチューニングに使用し、残りをチューニング済みモデルの評価に使用します。
2 番目のステップは、チューニング ジョブが読み込めるように、データセットを 1 行に 1 つの例を含む JSONL 形式に変換することです。


次に、クエリを実行してデータセットを DataFrame `df` にまとめましょう:


In [ ]:
%%bigquery df

SELECT CONCAT(q.title, q.body) as input_text, a.body AS output_text
FROM
    `bigquery-public-data.stackoverflow.posts_questions` q
JOIN
    `bigquery-public-data.stackoverflow.posts_answers` a
ON
    q.accepted_answer_id = a.id
WHERE
    q.accepted_answer_id IS NOT NULL AND
    REGEXP_CONTAINS(q.tags, "python") AND
    a.creation_date >= "2020-01-01"
LIMIT
    1000

In [ ]:
df.head()

`parts` の下の `input_text` 列は StackOverflow ユーザーが実際に投稿した質問に対応し、`output_text` 列は正解の回答に対応します。この 1,000 件の質問と回答のペアのデータセットから、以下の形式で 1 行に 1 つの例を含む JSONL ファイルを生成する必要があります:

```python
{
  "contents": [
    {
      "role": "user",
      "parts": [
        {
          "text": input_text,
          ...
        }
      ],
    },
    {
      "role": "model",
      "parts": [
        {
          "text": output_text,
          ...
        }
      ],
    },
  ]
}
```

これは Gemini 3.8 Flash モデルのチューニングに必要な形式の例です。

使用できるその他のフィールドについては、[ドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/tuning/supervised-tuning/prepare-data) を参照してください。

Gemini のチューニングには、少なくとも 100〜500 個の例を用意することをお勧めします。データセットで提供する例が多いほど、結果は良くなります。トレーニング データセットの例の数に制限はありません。ここでは 800 個を使用します。
可能であれば、検証データセットも提供してください。検証データセットは、チューニング ジョブの有効性を測定するのに役立ちます。検証データセットは最大 256 個の例をサポートしています。

まず、データをトレーニング用と評価用に分割しましょう。


In [ ]:
# 80/20 の比率で分割
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
evaluation = df.iloc[split_idx:]
print("train size:", len(train))
print("eval size:", len(evaluation))


In [ ]:
train.head()

チューニングを行うには、まずトレーニング データと評価データを JSONL 形式に変換する必要があります。そのために、次の 2 つのヘルパー関数を提供します。
最初の関数は、1 つの `input_text` と `output_text` のレコードを Gemini で必要な JSONL 形式に変換します。


### 演習

下のセルで関数の本体を完成させ、上記の Gemini 形式と互換性のある JSON 行を返すようにしてください。

**ヒント:** Python 辞書を JSON 文字列に変換するには `json.dumps` を使用します。


In [ ]:
def format_for_gemini(input_text, output_text):
    jsonl = None  # TODO: ここを補完してください。
    return jsonl


2 つ目のヘルパー関数はデータをファイルにエクスポートします:


In [ ]:
def export_tuning_data(file_name, df):
    with open(file_name, "a") as file:
        for row in df.iterrows():
            jsonline = format_for_gemini(
                row[1]["input_text"],
                row[1]["output_text"],
            )
            file.write(jsonline)
            file.write("\n")

それでは、トレーニング ファイルと評価ファイルを作成しましょう:


In [ ]:
training_data_filename = "tune_data_stack_overflow_python_qa.jsonl"
evaluation_data_filename = "evaluation_data_stack_overflow_python_qa.jsonl"

!test -f $training_data_filename    && rm $training_data_filename
!test -f $evaluation_data_filename  && rm $evaluation_data_filename

export_tuning_data(training_data_filename, train)
export_tuning_data(evaluation_data_filename, evaluation)

その後、ローカル ファイルを Cloud Storage（GCS）にエクスポートして、Agent Platform のチューニング ジョブで使用できるようにします。


In [ ]:
!gcloud storage cp $training_data_filename   $BUCKET_URI
!gcloud storage cp $evaluation_data_filename $BUCKET_URI

ファイルが Google Cloud Storage バケットに正常に転送されたことを確認できます:


In [ ]:
TRAINING_DATA_URI = f"{BUCKET_URI}/{training_data_filename}"
EVALUATION_DATA_URI = f"{BUCKET_URI}/{evaluation_data_filename}"

!gcloud storage ls -a -l $TRAINING_DATA_URI
!gcloud storage ls -a -l $EVALUATION_DATA_URI

### モデルのチューニング
それでは、モデルのチューニングを開始しましょう。[Google Gen AI SDK を使用してチューニング ジョブを送信](https://googleapis.github.io/python-genai/#tune) します。
これにはおよそ 30 分ほどかかります。

**演習:**<br>
[ドキュメント](https://googleapis.github.io/python-genai/#tune) を参考に以下のコードを補完し、チューニング ジョブを実行してください。


In [ ]:
client = genai.Client(vertexai=True, location="us-central1")

base_model = "gemini-3.8-flash"

training_dataset = types.TuningDataset(
    gcs_uri=TRAINING_DATA_URI,
)
evaluation_dataset = types.TuningValidationDataset(gcs_uri=EVALUATION_DATA_URI)

sft_tuning_job = client.tunings.tune(
    # TODO
)

running_states = {"JOB_STATE_PENDING", "JOB_STATE_RUNNING"}

# ジョブ完了のポーリング
while sft_tuning_job.state in running_states:
    print(sft_tuning_job.state)
    sft_tuning_job = client.tunings.get(name=sft_tuning_job.name)
    time.sleep(10)

print(sft_tuning_job.tuned_model_display_name)
print(sft_tuning_job.name)
print(sft_tuning_job.experiment)


## Agent Platform Model Registry からのチューニング済みモデルの取得


チューニング ジョブが完了すると、Agent Platform 上でモデルが利用可能になります。次のセルは、チューニング済みモデルを一覧表示する方法を示しています。


In [ ]:
for model in client.models.list(config={"page_size": 10, "query_base": False}):
    print(model)

### 演習

予測を取得してみましょう。Gen AI SDK を介してチューニング済みモデルにプロンプトを送信できます。<br>

**演習:**
以下の `model` フィールドを補完し、チューニング済みモデルを呼び出してください。


In [ ]:
# 推論のためにロケーションをマルチリージョンに切り替え
client = genai.Client(vertexai=True, location="us")

PROMPT = """
How can I store my TensorFlow checkpoint on Google Cloud Storage?

Python example:

"""

response = client.models.generate_content(
    model=...,  # TODO
    contents=PROMPT,
)

Markdown(response.text)


## 手動評価

モデルのパフォーマンスを把握するには、モデルを評価することが不可欠です。評価は、F1、BLEU、ROUGE などの評価指標を使用して自動化された方法で行うことができます。また、人間による評価手法を活用することもできます。人間による評価手法では、LLM の回答の質を人間に評価してもらいます。これはクラウドソーシングを通じて行うことも、専門家に回答を評価してもらうこともできます。代表的な人間による評価指標には、流暢さ（fluency）、一貫性（coherence）、関連性（relevance）、情報量（informativeness）などがあります。多くの場合、モデルのパフォーマンスを的確に把握するために、複数の評価指標を組み合わせて選択します。


ここでは、自動化された手法でありながら 2 つのテキストがどの程度同じ意味を持っているかをおおまかに測定する、次の 2 つの指標を計算します: 
- [BLEU](https://en.wikipedia.org/wiki/BLEU) 評価指標は一種の**適合率（precision）**指標であり、生成された文の $n$-gram が参照文の $n$-gram と一致する割合を測定します。スコアは 0 から 1 までで、文が似ているほどスコアが高くなります。BLEU1 は 1-gram（uni-gram）のみを考慮し、BLEU2 は 2-gram（bi-gram）を考慮します。

- [ROUGE](https://en.wikipedia.org/wiki/ROUGE_(metric)) 評価指標は一種の**再現率（recall）**指標であり、参照文の $n$-gram が生成文の $n$-gram と一致する割合を測定します。スコアは 0 から 1 までで、文が似ているほどスコアが高くなります。ROUGE1 は 1-gram（uni-gram）のみを考慮し、ROUGE2 は 2-gram（bi-gram）を考慮します。


スコアの計算には [evaluate](https://github.com/huggingface/evaluate/tree/main) を使用します。
このノートブックの前半で、トレーニング データセットと評価データセットを作成しました。ここでは評価データの一部を使用します。質問を使用してチューニング済みモデルから応答を取得し、元の回答を参照として使用します:
- **Candidates（候補）**: チューニング済みモデルによって生成された回答。
- **References（参照）**: 比較のために使用する元の回答。


In [ ]:
# 使用する行数を変更できます
EVAL_ROWS = 60
INPUT_LIMIT = 10000  # characters
evaluation = evaluation[evaluation.input_text.apply(len) <= INPUT_LIMIT]
evaluation = evaluation.head(EVAL_ROWS)
evaluation.head()


### 演習

下のセルの関数は、`evaluation.input_text` を使用してチューニング済みモデルにクエリを実行し、正解データをモデルの回答の横にある DataFrame 内の `evaluation.output_text` に保存します。

以下のコードを完成させてください:


In [ ]:
def create_eval_data(model_endpoint, evaluation):
    model_answers = []

    for prompt in evaluation.input_text:
        # TODO

    eval_df = pd.DataFrame(
        {"candidate": model_answers, "reference": evaluation.output_text}
    )
    mask = eval_df.candidate == ""
    return eval_df[~mask]

In [ ]:
eval_df = create_eval_data(sft_tuning_job.tuned_model.endpoint, evaluation)

In [ ]:
eval_df.head()

次のセルの関数は、1-gram の BLEU および ROUGE スコアを計算します。これらを参照回答とチューニング済みモデルによって生成された回答全体で平均化し、モデルのパフォーマンス指標となるスコアを算出します。


In [ ]:
def compute_scores(eval_data):
    predictions = eval_data.candidate.tolist()
    references = eval_data.reference.tolist()
    rouge = evaluate.load("rouge")
    bleu = evaluate.load("bleu")
    rouge_value = rouge.compute(predictions=predictions, references=references)[
        "rouge1"
    ]
    bleu_value = bleu.compute(predictions=predictions, references=references)[
        "bleu"
    ]
    return {"rouge": rouge_value, "bleu": bleu_value}

In [ ]:
compute_scores(eval_df)

モデルの 2 つのバージョン（異なるデータ量やトレーニング ステップでチューニングされたものなど）がある場合、スコアを比較してどちらが優れているかを判断できるようになります。ただし、これらの自動評価指標は、人間による評価のおおまかな代用指標にすぎない点に注意してください。


## 自動評価


最後に、Agent Platform のチューニング ジョブはモデル チューニングとモデル評価の指標を収集してレポートすることに触れておきます。これらは、Agent Platform Studio のチューニング セクションでチューニング済みモデル名をクリックすることで、Agent Platform Experiments で可視化できます。
計算される指標の説明は以下のとおりです:


#### モデル チューニング指標

モデル チューニング ジョブは、`gemini-3.8-flash` に対して次のチューニング指標を自動的に収集します。

* `/train_total_loss`: トレーニング ステップにおけるチューニング データセットの損失。
* `/train_fraction_of_correct_next_step_preds`: トレーニング ステップにおけるトークン正解率。単一の予測は一連のトークンで構成されます。この指標は、チューニング データセット内の正解と比較した、予測されたトークンの精度を測定します。
* `/train_num_predictions`: トレーニング ステップにおける予測トークンの数。


#### モデル検証指標:

このラボで行ったように評価データセットを渡すことで、gemini-3.8-flash に対する次の検証指標を収集するようにモデル チューニング ジョブを設定できます。

* `/eval_total_loss`: 検証ステップにおける検証データセットの損失。
* `/eval_fraction_of_correct_next_step_preds`: 検証ステップにおけるトークン正解率。単一の予測は一連のトークンで構成されます。この指標は、検証データセット内の正解と比較した、予測されたトークンの精度を測定します。
* `/eval_num_predictions`: 検証ステップにおける予測トークンの数。


指標の可視化は、モデル チューニング ジョブの完了後に利用可能になります。
チューニング ジョブの作成時に検証データセットを指定しなかった場合、
チューニング指標の可視化のみが利用可能です。


Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.